In [4]:
def process(dest_tm_1, dest_column_value_1, dest_tm_2, dest_column_value_2, dest_tm_3, dest_column_value_3, dest_tm_4, dest_column_value_4, min_period, profit, loss, time_in_ms, order_type, target_df_lst):
    try:
        # print('profit ', profit, ', loss ', loss, ', target_time_in_ms ', time_in_ms, ', order_type ', order_type, ', dest_tm ', dest_tm, ', dest_column_value ', dest_column_value)
        merged_list = []
        for idx in range(len(merged_files)):
            assert features_df_lst[idx].shape[0] == target_df_lst[idx].shape[0]
            merged = pd.concat([features_df_lst[idx], target_df_lst[idx]], axis=1).copy()

            merged['calculated_feature_1'], merged['calculated_feature_2'], merged['calculated_feature_3'], merged['calculated_feature_4'] = calculate_feature(merged.copy(), dest_tm_1, dest_tm_2, dest_tm_3, dest_tm_4, min_period)
            merged = filter_df(merged, dest_column_value_1, dest_column_value_2, dest_column_value_3, dest_column_value_4)

            merged_filtered = merged[merged.target!=0]
            # remove outliers
            quantile_const = 0.999
            quantile_val = merged_filtered.target.quantile(quantile_const)
            merged_filtered = merged_filtered[merged_filtered.target < quantile_val]

            merged_list.append(merged_filtered)

        filtered_trades_df = pd.DataFrame()
        all_trades_df = pd.DataFrame()
        for merged in merged_list:
            row_list = filter_trades_by_single_volume(merged.target.values, merged.start.values, merged.end.values)
            filtered_trades_df = pd.concat([filtered_trades_df, merged.iloc[row_list]], ignore_index=True, axis=0).copy()
            all_trades_df = pd.concat([all_trades_df, merged], ignore_index=True, axis=0).copy()
        filtered_trades_df['cumsum'] = filtered_trades_df.target.cumsum()

        if len(filtered_trades_df) > 30:
            # Calculate Sharpe Ratio
            mean_return = filtered_trades_df['target'].mean()
            std_return = filtered_trades_df['target'].std(ddof=1)
            sharpe_ratio = (mean_return / std_return) # Sharpe ratio

            # Calculate required sample size for 80% power
            from statsmodels.stats.power import TTestPower
            effect_size = mean_return / std_return  # Cohen's d
            alpha = 0.05  # Significance level
            power = 0.8   # Desired power
            power_analysis = TTestPower()
            required_n = power_analysis.solve_power(effect_size=effect_size, alpha=alpha, power=power, alternative='larger')
            if isinstance(required_n, np.ndarray):
                required_n = int(required_n[0])
            else:
                required_n = int(required_n)

            # print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
            # print(f"Required number of trades for 80% power: {required_n}")

            hv.extension('bokeh')
            # trades_df.hvplot.scatter(x='ft_received', y='cumsum')
            s = hv.Scatter(filtered_trades_df, 'ft_received', 'cumsum').opts(width=500, height=400)
            prefix = '_' if sharpe_ratio > 0 else ''
            file_name = f'{prefix}sharpe_{sharpe_ratio:.2f}_required_n_{required_n}_ttl_sum_{all_trades_df.target.cumsum().values[-1]}_cumsum_{filtered_trades_df.target.cumsum().values[-1]}_n_single_observation_{filtered_trades_df.shape[0]}_profit_{profit}_loss_{loss}_target_time_in_ms_{time_in_ms}_order_type_{order_type}_tm1_{dest_tm_1}_val1_{dest_column_value_1}_tm2_{dest_tm_2}_val2_{dest_column_value_2}_tm3_{dest_tm_3}_val3_{dest_column_value_3}_tm4_{dest_tm_4}_val4_{dest_column_value_4}_min_period_{min_period}'
            hv.save(s, path + file_name + '.auto', fmt='auto')
            # filtered_trades_df.to_csv(path + file_name + '.csv', sep=';')
    except Exception:
        os.write(1, ('error ' + ' profit ' + str(profit) + ', loss ' + str(loss) + ', target_time_in_ms ' + str(time_in_ms) + ', order_type ' + str(order_type) + ', dest_tm_1 ' + str(dest_tm_1) + ', dest_column_value_1 ' + str(dest_column_value_1) + ', dest_tm_2 ' + str(dest_tm_2) + ', dest_column_value_2 ' + str(dest_column_value_2) + ', dest_tm_3 ' + str(dest_tm_3) + ', dest_column_value_3 ' + str(dest_column_value_3) + ', dest_tm_4 ' + str(dest_tm_4) + ', dest_column_value_4 ' + str(dest_column_value_4) + ' ' + str(time.time()) + '\n').encode())
        print('Error')
        # raise Exception

def _f(pld):
    profit = pld[0]
    loss = pld[1]
    time_in_ms = pld[2]
    order_type = pld[3]

    target_df_lst = []
    for idx in range(len(merged_files)):
        df2 = pd.read_csv(f'target//{merged_files[idx]}_p{profit}_l{loss}_d{direction}_t{time_in_ms}_o{order_type}.csv', sep=';', engine='pyarrow')
        df2.rename(columns={df2.columns[0]: 'target', df2.columns[1]: 'start', df2.columns[2]: 'end'}, inplace=True)
        target_df_lst.append(df2)


# Parallel(n_jobs=8, verbose=10)(delayed(_f)(pld) for pld in itertools.product(profits, losses, target_timers_in_ms, order_types))

for pld in itertools.product(profits, losses, target_timers_in_ms, order_types):
    _f(pld)

KeyboardInterrupt: 